In [1]:
import pandas as pd
import numpy as np
from scipy.signal import butter, filtfilt

# ==========================================
# 0. 准备滤波器
# ==========================================
def butter_bandpass_filter(data, lowcut=0.5, highcut=10.0, fs=100.0, order=4):
    nyq = 0.5 * fs
    low = lowcut / nyq
    high = highcut / nyq
    b, a = butter(order, [low, high], btype='band')
    y = filtfilt(b, a, data)
    return y

# ==========================================
# 1. 数据加载与预处理
# ==========================================
print("正在加载和预处理数据...")
df = pd.read_csv("../data/raw/may_filtered.csv")
datetimestr = df['date'] + ' ' + df['time']
df['datetime'] = pd.to_datetime(datetimestr, format='%d-%m-%Y %H:%M:%S')

df['duration(s)'] = pd.to_timedelta(df.duration).dt.total_seconds()
df.drop(['date', 'time', 'duration'], inplace=True, axis=1)
df['motion'] = np.linalg.norm(df[['accX', 'accY', 'accZ']].values, axis=1)
df = df[["datetime", "duration(s)", "red", "ied", "motion"]]

# 提取全局 motion 的范围，用于后续固定 Y 轴
global_motion_min = df['motion'].min()
global_motion_max = df['motion'].max()
motion_ylim_bottom = global_motion_min - (global_motion_max - global_motion_min) * 0.05
motion_ylim_top = global_motion_max + (global_motion_max - global_motion_min) * 0.05

# 使用稳定排序
df = df.sort_values('datetime', kind='stable').reset_index(drop=True)

# 计算 magnitude
died = df['ied'].diff().abs()
safe_died = np.where((died == 0) | (died.isna()), 1, died)
df['magnitude'] = np.floor(np.log10(safe_died)).astype(int)

# ==========================================
# 2. 剔除异常点并进行片段强行分割 (双重切刀)
# ==========================================
is_mag_6 = df['magnitude'] == 6
prev_is_mag_6 = is_mag_6.shift(-1, fill_value=False)
to_drop = is_mag_6 | prev_is_mag_6

# 第一把刀：时间断层
time_diff = df['datetime'].diff().dt.total_seconds()
is_time_gap = time_diff > 1.0

# 第二把刀：强行跨越异常点
is_after_anomaly = (~to_drop) & (to_drop.shift(1, fill_value=False))

cut_points = is_time_gap | is_after_anomaly
group_ids = cut_points.cumsum()

df_splits = []
df_temp = df.assign(group_id=group_ids, to_drop_flag=to_drop)

for _, group in df_temp.groupby('group_id'):
    clean_chunk = group[~group['to_drop_flag']].drop(columns=['group_id', 'to_drop_flag'])
    if not clean_chunk.empty:
        clean_chunk = clean_chunk.reset_index(drop=True)
        df_splits.append(clean_chunk)

print(f"原数据共 {len(df)} 行，剔除了 {to_drop.sum()} 行 mag=6 相关的异常点。")
print(f"根据时间断层和异常点截断，成功切分出 {len(df_splits)} 个纯净片段。")

# ==========================================
# 3. 统计各片段信息
# ==========================================
summary_results = []
for i, chunk in enumerate(df_splits):
    if chunk.empty: continue
    dt_values = chunk['datetime'].values 
    unique_dt = np.unique(dt_values) 
    total_duration = 0 if len(unique_dt) < 2 else (unique_dt[-1] - unique_dt[0]) / np.timedelta64(1, 's')
    mags = chunk['magnitude'].values
    summary_results.append({
        '片段序号': i + 1, '数据行数': len(chunk), '总时长(s)': total_duration,
        'mag=4_个数': np.sum(mags == 4), 'mag=5_个数': np.sum(mags == 5), 'mag=6_个数': np.sum(mags == 6)
    })

df_mag_summary = pd.DataFrame(summary_results)

# 筛选并排序准备可视化的数据
valid_chunks = [
    {'original_idx': row['片段序号'], 'duration': row['总时长(s)'], 'data': df_splits[i]}
    for i, row in df_mag_summary.iterrows() if row['总时长(s)'] > 60.0
]

# 按时长从小到大排序
# valid_chunks = sorted(valid_chunks, key=lambda x: x['duration'])
print(f"数据处理完毕。准备可视化：共有 {len(valid_chunks)} 个片段时长 > 60 秒 。")

正在加载和预处理数据...
原数据共 7893800 行，剔除了 531 行 mag=6 相关的异常点。
根据时间断层和异常点截断，成功切分出 410 个纯净片段。
数据处理完毕。准备可视化：共有 126 个片段时长 > 60 秒 。


In [2]:
#   valid_chunks    motion_ylim_bottom    motion_ylim_top

In [3]:
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display
from matplotlib import rcParams

# 设置字体，防止中文乱码
rcParams['font.family'] = 'SimHei'
rcParams['axes.unicode_minus'] = False

# ==========================================
# 4. 可视化模块
# ==========================================
if valid_chunks:
    def view_segment(segment_idx, start_row, window_size, use_filter):
        chunk_info = valid_chunks[segment_idx]
        df_plot = chunk_info['data']
        total_rows = len(df_plot)
        
        if start_row >= total_rows: start_row = max(0, total_rows - window_size)
        end_row = min(total_rows, start_row + window_size)
        
        plot_data = df_plot.iloc[start_row:end_row]
        if plot_data.empty: return
            
        ied_array = plot_data['ied'].values
        motion_array = plot_data['motion'].values
        
        if use_filter and len(ied_array) > 15:
            try: 
                ied_array = butter_bandpass_filter(ied_array)
            except ValueError: 
                pass 
                
        fig, ax1 = plt.subplots(figsize=(15, 5)) 
        x_axis = np.arange(start_row, end_row)
        
        # 绘制 IED
        color_ied = '#1f77b4'
        line1 = ax1.plot(x_axis, ied_array, color=color_ied, linewidth=1.2, label='IED 信号 (左轴)')
        ax1.set_xlabel("数据点绝对序号 (平铺展示)")
        ax1.set_ylabel("IED 信号幅值", color=color_ied)
        ax1.tick_params(axis='y', labelcolor=color_ied)
        ax1.set_xlim(start_row, end_row)
        ax1.grid(True, linestyle='--', alpha=0.6)
        
        # 绘制 Motion 并锁定 Y 轴
        ax2 = ax1.twinx()  
        color_motion = '#ff7f0e' 
        line2 = ax2.plot(x_axis, motion_array, color=color_motion, linewidth=1.5, alpha=0.7, label='Motion 强度 (右轴)')
        ax2.set_ylabel("Motion (加速度模长)", color=color_motion)
        ax2.tick_params(axis='y', labelcolor=color_motion)
        
        # 使用第一部分计算的全局 Y 轴极值
        ax2.set_ylim(motion_ylim_bottom, motion_ylim_top)
        
        lines = line1 + line2
        labels = [l.get_label() for l in lines]
        ax1.legend(lines, labels, loc='upper right')
        
        start_time_str = plot_data['datetime'].iloc[0].strftime('%H:%M:%S')
        end_time_str = plot_data['datetime'].iloc[-1].strftime('%H:%M:%S')
        title_str = (f"已排序序号: {segment_idx} | 原始片段序号: {chunk_info['original_idx']} | 时长: {chunk_info['duration']:.2f}s | 总行数: {total_rows}\n"
                     f"时间跨度: {start_time_str} - {end_time_str}")
        if use_filter: title_str += " (已开启 0.5-10Hz IED 带通滤波)"
            
        plt.title(title_str)
        plt.tight_layout() 
        plt.show()

    # 设置交互组件
    segment_slider = widgets.IntSlider(min=0, max=len(valid_chunks)-1, step=1, value=0, description='切换片段(按时长):', layout=widgets.Layout(width='600px'))
    start_slider = widgets.IntSlider(min=0, max=300000, step=100, value=0, description='起始行号:', layout=widgets.Layout(width='600px'))
    window_slider = widgets.IntSlider(min=200, max=5000, step=100, value=1000, description='窗口(点数):', layout=widgets.Layout(width='600px'))
    filter_checkbox = widgets.Checkbox(value=True, description='开启基线滤波 (强烈推荐)', indent=False)
    
    # 动态更新起始行号的最大值
    def update_start_slider(*args):
        total_rows = len(valid_chunks[segment_slider.value]['data'])
        start_slider.max = max(0, total_rows - window_slider.value)
        
    segment_slider.observe(update_start_slider, 'value')
    window_slider.observe(update_start_slider, 'value')
    update_start_slider() 

    # 启动交互
    widgets.interact(view_segment, segment_idx=segment_slider, start_row=start_slider, window_size=window_slider, use_filter=filter_checkbox)
else:
    print("没有符合条件的数据可供可视化。")

interactive(children=(IntSlider(value=0, description='切换片段(按时长):', layout=Layout(width='600px'), max=125), Int…

In [6]:
import neurokit2 as nk
import numpy as np
import pandas as pd
from scipy.interpolate import interp1d

print("开始进行时间轴重建、插值重采样与 NeuroKit2 特征提取...")

# 强制目标采样率
TARGET_FS = 100

# ==========================================
# ★ 指定你要测试的片段索引 ★
# ==========================================
target_index = 0 

if target_index < len(valid_chunks):
    chunk_info = valid_chunks[target_index]
    chunk = chunk_info['data'].copy()
    
    # ==========================================
    # 步骤 1: 重建精确时间轴 (微秒级对齐)
    # ==========================================
    # 计算每一秒内包含的数据行数
    group_counts = chunk.groupby('datetime')['datetime'].transform('count')
    # 计算当前行在这一秒内排第几个 (0 到 count-1)
    group_rank = chunk.groupby('datetime').cumcount()
    
    # 给同一秒内的数据分配均匀的小数秒偏移量 (例如 0.0, 0.01, 0.02...)
    fractional_seconds = group_rank / group_counts
    
    # 建立一个相对起始点的连续浮点秒数时间轴 (time_sec)
    start_time = chunk['datetime'].iloc[0]
    chunk['time_sec'] = (chunk['datetime'] - start_time).dt.total_seconds() + fractional_seconds
    
    # ==========================================
    # 步骤 2: 线性插值重采样到完美的 100Hz
    # ==========================================
    # 创建一个绝对完美的、间隔严格为 0.01s 的时间网格
    total_duration = chunk['time_sec'].iloc[-1]
    perfect_time_grid = np.arange(0, total_duration, 1.0 / TARGET_FS)
    
    # 创建一个新的 DataFrame 来存放重采样后的纯净数据
    df_resampled = pd.DataFrame({'time_sec': perfect_time_grid})
    
    # 还原对应的绝对时间 datetime (方便你后续画图)
    df_resampled['datetime'] = start_time + pd.to_timedelta(perfect_time_grid, unit='s')
    
    # 将需要的列全部插值过去
    cols_to_interp = ['ied', 'red', 'motion']
    original_time = chunk['time_sec'].values
    
    for col in cols_to_interp:
        if col in chunk.columns:
            # fill_value="extrapolate" 防止首尾越界报错
            interpolator = interp1d(original_time, chunk[col].values, kind='linear', fill_value="extrapolate")
            df_resampled[col] = interpolator(perfect_time_grid)
    
    # ==========================================
    # 步骤 3: 喂给 NeuroKit2 (此时信号已经是完美的 100Hz)
    # ==========================================
    raw_signal = df_resampled['ied'].values
    
    try:
        # 清洗信号
        cleaned_signal = nk.ppg_clean(raw_signal, sampling_rate=TARGET_FS)
        df_resampled['ied_cleaned'] = cleaned_signal
        
        # ==========================================
        # ★ 修正处：使用支持同时检测波谷的算法（如 charlton 或 bishop）
        # 一次性返回波峰和波谷，无需 delineate
        # ==========================================
        info_peaks = nk.ppg_findpeaks(cleaned_signal, sampling_rate=TARGET_FS, method="charlton")
        
        # 从字典中提取波峰和波谷
        peaks = info_peaks["PPG_Peaks"]
        troughs = info_peaks.get("PPG_Onsets", [])  # 使用 .get 提取波谷更安全
        
        # 标记波峰与波谷
        df_resampled['is_peak'] = 0
        df_resampled['is_trough'] = 0
        
        # 过滤掉可能的 NaN 值，并转换为整型索引
        valid_peaks = [int(p) for p in peaks if not np.isnan(p)]
        valid_troughs = [int(t) for t in troughs if not np.isnan(t)]
        
        df_resampled.loc[valid_peaks, 'is_peak'] = 1
        df_resampled.loc[valid_troughs, 'is_trough'] = 1
        
        # 【重点】将原始不均匀的数据替换为完美的重采样数据
        chunk_info['data'] = df_resampled
        
        print(f"--> 成功！片段 {target_index} (原始序号: {chunk_info['original_idx']}) 时间重构与特征提取完成。")
        print(f"--> 原数据行数: {len(chunk)} ➡️ 重采样后标准行数: {len(df_resampled)} (严格 {TARGET_FS}Hz)")
        print(f"--> 共提取到 {len(valid_peaks)} 个波峰，{len(valid_troughs)} 个波谷。")
        
    except Exception as e:
        print(f"--> 错误！片段 {target_index} 处理失败: {e}")

else:
    print(f"--> 越界错误：你指定的片段索引 {target_index} 不存在。")

开始进行时间轴重建、插值重采样与 NeuroKit2 特征提取...
--> 成功！片段 0 (原始序号: 1.0) 时间重构与特征提取完成。
--> 原数据行数: 36780 ➡️ 重采样后标准行数: 18500 (严格 100Hz)
--> 共提取到 215 个波峰，206 个波谷。
